In [1]:
import os, glob
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from scipy import stats
%matplotlib qt

In [31]:
# Import data file
# data_path = 'C:/Users/mvmigem/Documents/data/project_2/pilot/hannah/participant_94.csv'
data_path = 'C:/Users/mvmigem/Documents/data/project_2/pilot/gauthier/participant_85.csv'
df_behaviour = pd.read_csv(data_path)
df_behaviour = df_behaviour[~df_behaviour['block'].isin([1,4])]


In [42]:
bleh = df_behaviour[(df_behaviour['target_trials'] == 1)]
bleh = bleh[(bleh['sequence'] == 1)]
bleh['acc_simple'] = np.where(bleh['accuracy'] == 'correct', 'correct', 'wrong')

blehA = bleh[bleh['task'] == 'angle']
blehR= bleh[bleh['task'] == 'rotation']
target_df = df_behaviour[~df_behaviour['accuracy'].isin(['correct_nogo'])]

In [43]:
sns.barplot(bleh,x='task',y='rt' )

<Axes: xlabel='Accuracy', ylabel='Percentage (%)'>

In [49]:


# Create the plot
# sns.countplot(data=blehA, x='accuracy', hue='rotation_prediction',stat='percent')
# sns.countplot(data=bleh, x='accuracy', hue='task',stat='percent')
sns.histplot(data=bleh, x='accuracy', hue='task', stat='percent', common_norm=False, multiple='dodge')

plt.xlabel('Accuracy')
plt.ylabel('Percentage (%)')
plt.show()


In [45]:
def calculate_d_prime(hits, misses, false_alarms, correct_rejections):
    """
    Calculate d-prime from signal detection theory counts
    
    Parameters:
    hits (int): Number of hits (correctly detected signals)
    misses (int): Number of misses (missed signals)
    false_alarms (int): Number of false alarms (false positives)
    correct_rejections (int): Number of correct rejections (true negatives)
    
    Returns:
    float: d-prime value
    """
    # Calculate hit rate and false alarm rate with edge protection
    hit_rate = hits / (hits + misses)
    false_alarm_rate = false_alarms / (false_alarms + correct_rejections)
    
    # Adjust rates to avoid 0 or 1 (which would give infinite z-scores)
    hit_rate = min(0.99, max(0.01, hit_rate))
    false_alarm_rate = min(0.99, max(0.01, false_alarm_rate))
    
    # Calculate z-scores
    z_hit = stats.norm.ppf(hit_rate)
    z_fa = stats.norm.ppf(false_alarm_rate)
    
    return z_hit - z_fa

In [46]:
# d-prime calculations for angle task
hits = ((blehA['angle_prediction'] == 'odd' ) & (blehA['acc_simple'] == 'correct')).sum()
misses = ((blehA['angle_prediction'] == 'odd' ) & (blehA['acc_simple'] == 'wrong')).sum()
false_alarm = ((blehA['angle_prediction'] == 'regular' ) & (blehA['acc_simple'] == 'wrong')).sum()
correct_rejections = ((blehA['angle_prediction'] == 'regular' ) & (blehA['acc_simple'] == 'correct')).sum()

angle_dprime = calculate_d_prime(hits,misses,false_alarm,correct_rejections)

In [47]:
# d-prime calculations for rotation task
hits = ((blehR['rotation_prediction'] == 'odd' ) & (blehR['acc_simple'] == 'correct')).sum()
correct_rejections = ((blehR['rotation_prediction'] == 'regular' ) & (blehR['acc_simple'] == 'correct')).sum()
false_alarm = ((blehR['rotation_prediction'] == 'regular' ) & (blehR['acc_simple'] == 'wrong')).sum()
misses = ((blehR['rotation_prediction'] == 'odd' ) & (blehR['acc_simple'] == 'wrong')).sum()

rotation_dprime = calculate_d_prime(hits,misses,false_alarm,correct_rejections)

In [51]:
sns.barplot(x=['angle','rotation'],y=[angle_dprime,rotation_dprime] )

<Axes: >